[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/pose_diffusion/station_keeping.ipynb)

# Station Keeping in Gusts

A vessel on dynamic positioning holds a fixed spot with its thrusters while wind and waves push it around. It never sits exactly on its set point. It wanders, and more sideways than forward, because gusts hit the side of the hull harder than the bow. That raises two practical questions: how much clear water does the vessel need around it, and how stiff does the thruster controller have to be? This notebook answers both. It predicts the vessel's spread directly, checks the prediction against a simulated fleet, and then uses it to choose a controller gain.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import PGA2D
from examples.geometry.pose_diffusion import render

np.set_printoptions(precision=3, suppress=True)

ga = PGA2D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Twist = ga.gatype.bivector()            # a small motion: a rotation and a translation together
Line = ga.gatype.antibivector()         # a line measures a twist: line & twist is a number
set_point = mv.xy                       # the set point, at the origin; in PGA2D a point is a bivector
print(Twist, Line, sep="\n")

## 1. Gusts and the controller

The vessel's error is a twist: the small motion that takes the set point to where the vessel actually is. Gusts push it at random. A gust is modelled as white noise sent through a kick map, a map from lines to twists, scaled by how hard gusts push sideways, forward and in yaw. The controller is a map on twists. It reduces the error by a fixed fraction per second, the gain, so the error decays with a time constant of one over the gain. If the vessel also turns on the spot, its error turns with it; the commutator with the turning rate adds that.

For comparison, in matrix notation one chooses an order for the twist's coordinates, say $(v_x, v_y, \omega)$, and writes the dynamics as $F = \operatorname{ad}_\Omega - k\,I$, with $\operatorname{ad}_\Omega$ the $3\times3$ adjoint matrix of the turning rate in that order, signs included, and the kick map as a $3\times3$ matrix $B$.

In [ ]:
# How hard gusts push, per square root of a second, in the vessel's frame: metres sideways and
# forward, radians in yaw. The side of the hull catches much more than the bow.
sideways, forward, yaw = 0.5, 0.15, 0.01
# One unit of white noise on each basis line becomes a push in one direction:
kicks = mv.yw * sideways * (mv.yw & Line) + mv.wx * forward * (mv.wx & Line) + mv.xy * yaw * (mv.xy & Line)   # Twist <- Line


def drift(turning: Twist, gain: Scalar):
    """How a position error changes per second: it turns with the vessel, and the controller shrinks it."""
    return Twist.commutator(turning) - gain * Twist                    # Twist <- Twist


still = mv.xy * 0.0                                                    # the vessel keeps its heading
controller = drift(still, mv.scalar([0.1]))                            # gain 0.1 per second: a 10 s time constant

## 2. The spread as a covariance

The spread of the vessel's position is a covariance. Here it is a map from lines to twists. A line measures the error, `line & error` is one number, and the covariance returns the twist that is correlated with that measurement; `line & covariance(line)` is the variance of the measurement. The covariance the gusts add per second follows from the kick map. White noise has unit variance on each basis line and no correlation between them, so it is the sum over the basis of each line's kick joined with itself.

In [ ]:
basis = mv.basis()                                                        # [3] Line
noise = (kicks(basis) * (kicks(basis) & Line)).sum(axis=0)                # Twist <- Line: covariance added per second

As the errors change, so does their covariance. The controller acts on the twists the covariance returns. It also changes which errors a given line picks out: measuring the changed error `controller(t)` with a line gives the same number as measuring `t` itself with another line. That other line comes from solving the incidence form against the controller. In matrix language it is the transpose of the controller. The covariance's rate of change is the sum of these two effects and the gusts.

For comparison, in matrix notation the covariance is a symmetric $3\times3$ matrix $P$ and its rate of change is $\dot P = F P + P F^\top + Q$, with $Q = B B^\top$. The transpose $F^\top$ is the controller acting on measurements, the role `on_lines` plays here.

In [ ]:
def on_lines(dynamics):
    """The line that measures an error the way the given line measures the changed error."""
    return (Line & Twist).solve(Line & dynamics)                        # Line <- Line


def growth(dynamics, covariance):
    """Rate of change of the covariance: the dynamics on the twists it returns and on the lines it
    takes, plus what the gusts add."""
    return dynamics(covariance) + covariance(on_lines(dynamics)) + noise

## 3. The steady state in one solve

After a while the controller removes spread as fast as the gusts add it, and the covariance stops growing. The growth is linear in the covariance, so this balance is a linear equation whose unknown is a map. Write the covariance as a sum of dyads, a twist times a measurement of another twist, and leave all three open. The growth then becomes a map with three inputs. `lstsq` matches its line input against the gusts and solves for the other two at once. The result takes a line through its dual, the twist that measures the same as the line.

For comparison, in matrix notation this is the continuous Lyapunov equation $F P + P F^\top + Q = 0$. It is solved either by vectorizing, $(I \otimes F + F \otimes I)\,\operatorname{vec} P = -\operatorname{vec} Q$, a $9\times9$ system whose matrix is `balance` with its two twist slots flattened into one, or by the Bartels–Stewart algorithm, `scipy.linalg.solve_continuous_lyapunov`.

In [ ]:
def settled(dynamics):
    """The covariance at which growth stops."""
    balance = dynamics(Twist) * (Twist & Line) + Twist * (dynamics(Twist) & Line)   # Twist <- (Twist, Twist, Line)
    return balance.lstsq(-noise)(Line.dual())                            # Twist <- Line


steady = settled(controller)                                             # Twist <- Line

## 4. Checking against a fleet

To check the prediction, simulate 2000 vessels for two and a half minutes. In each step every vessel's error changes under the controller and takes its own gust, and the predicted covariance is advanced by its growth over the same step. White noise over a step has a spread proportional to the square root of the step's length, so the gusts are scaled by the square root of dt. The figure shows the vessels, each as a short arrow along its heading, with the integrated prediction (solid) and the steady state from the solve (dashed) as 2σ ellipses of position.

In [ ]:
def simulate(dynamics, vessels: int, seconds: float, dt: float, seed: int):
    """A simulated fleet and the predicted covariance of its errors, from all vessels on the set point."""
    rng = np.random.default_rng(seed)
    errors = mv.bivector(np.zeros((vessels, 3)))                        # [vessels] Twist: all on the set point
    predicted = noise * 0.0                                             # no spread at the start
    for _ in range(int(seconds / dt)):
        white = mv.vector(rng.normal(size=(vessels, 3))) * np.sqrt(dt)  # [vessels] Line: white noise over the step
        errors = errors + dynamics(errors) * dt + kicks(white)
        predicted = predicted + growth(dynamics, predicted) * dt
    return errors, predicted


errors, predicted = simulate(controller, 2000, 150.0, 0.2, 0)
poses = (errors * 0.5).exp()                                           # [vessels] Motor: each vessel's pose
render.draw_settled({"holding heading, gain 0.1 /s": (poses, predicted, steady)}, 5.0);

## 5. Choosing the controller gain

A stiffer controller keeps the vessel closer, at the cost of more thruster work. Because the steady state is a single solve, it can be computed for a whole range of gains at once: the gain becomes a batch of scalars, and every map above broadcasts over it. The position's spread then follows by measuring how an error moves the set point. Suppose the vessel must keep its 2σ excursion within 2 m in every direction. The plot reads off the smallest gain that does it.

For comparison, in matrix notation the position covariance is $\Sigma = J P J^\top$, with $J$ the $2\times3$ Jacobian of the set point's displacement with respect to the twist coordinates; here, with the set point on the rotation centre, it picks out the two translation coordinates.

In [ ]:
def position(covariance):
    """Covariance of position measurements: a line measures how an error moves the set point."""
    moves = Twist.commutator(set_point)                                 # Point <- Twist: how an error moves the set point
    through = (Line & Twist).solve(Line & moves)                       # Line <- Line: a position measurement as a twist measurement
    return through & covariance(through)                               # Scalar <- (Line, Line)


gains = np.linspace(0.02, 0.5, 60)                                     # per second
spreads = position(settled(drift(still, mv.scalar(gains[:, None]))))   # [gains] Scalar <- (Line, Line)
render.draw_envelope(gains, spreads, 2.0);

Along the vessel's side the 2σ excursion is the gust strength over the square root of twice the gain, so the 2 m tolerance needs a gain of 0.125 per second, an 8 s time constant. Forward, where gusts push less, the same gain leaves a lot of margin.

## 6. Turning on the spot

A vessel that turns while it holds position carries its error around with it. A sideways push becomes a forward error a quarter turn later, and gusts push less in that direction. The commutator with the turning rate adds exactly this to the dynamics, and nothing else in the calculation changes. At one turn a minute the spread comes out rounder and a little smaller along its long axis.

In [ ]:
turning = drift(mv.xy * (2 * np.pi / 60), mv.scalar([0.1]))           # one full turn a minute
turning_errors, turning_predicted = simulate(turning, 2000, 150.0, 0.2, 1)
render.draw_settled({"holding heading": (poses, predicted, steady),
                     "one turn a minute": ((turning_errors * 0.5).exp(), turning_predicted, settled(turning))}, 5.0);

In [ ]:
# checks
# The steady state does not grow, the integrated prediction reaches it, and the fleet matches it.
np.testing.assert_allclose(growth(controller, steady).kernel, 0.0, atol=1e-10)
np.testing.assert_allclose(predicted.kernel, steady.kernel, atol=1e-6)
sample = (errors * (errors & Line)).mean(axis=0)                       # Twist <- Line
np.testing.assert_allclose(sample.kernel, steady.kernel, atol=0.1 * np.abs(steady.kernel).max())
# Sideways the variance is the gust strength squared over twice the gain.
np.testing.assert_allclose(np.sort(position(steady).eigvals().real().to_array())[-2], sideways**2 / 0.2, rtol=1e-8)